# ArNet2 - Preprocessing and Model Training

This notebook demonstrates the workflow for:
1. Preprocessing ECG signal data from the **SHDB-AF** dataset.
2. Training the **ArNet2** model for classification of **Atrial Fibrillation (AF)**.

### Disclaimer:

The **preprocessed data** in this notebook is **constructed from only one patient** from the **SHDB-AF** dataset. In real-world applications, **more diverse training data** (from multiple patients with varying AF conditions) is required to build a robust model. Using data from a single patient can result in overfitting and limit the generalization ability of the model. Therefore, for a model to perform well on unseen data, it is essential to train on a **larger and more varied dataset**.

### Dataset Overview:

The **SHDB-AF** dataset consists of ECG signals, each annotated with:
- **R-peak annotations**: The location of R-peaks in the ECG signal.
- **AF labels per peak**: Whether the peak is associated with **Atrial Fibrillation (AF)**.
- **Overall patient-level label**: The AF status of the patient (e.g., **PAF**: Paroxysmal AF, **Persistent AF**, **Non-AF**).

This dataset will be used to demonstrate how to preprocess the data and train the **ArNet2** model.


### 1. Import Libraries

In [2]:
import os
import numpy as np
import pandas as pd
import wfdb
import requests
import pickle
import subprocess


### 2. Setting Up Paths and Directories

Before proceeding, we set up the paths where data will be downloaded and processed:

In [3]:
# Setting up the paths
data_path = '.././physionet_data'  # Where you'll download the PhysioNet dataset
output_path = '.././data'  # Where to save the processed data
os.makedirs(data_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

### 3. Download Data

In this section, we define a function to download necessary files from PhysioNet. These files include the ECG signal and annotations:

In [4]:
def download_file(url, filename):
    """
    Downloads a file from a specified URL and saves it to the given filename.

    :param url: URL to the file to be downloaded.
    :param filename: Path where the downloaded file will be saved.
    """
    response = requests.get(url)
    with open(filename, 'wb') as f:
        f.write(response.content)

# Example download for a single record
record_name = '001'
filepath = f'{data_path}/{record_name}'
url = f'https://physionet.org/files/shdb-af/1.0.1/{record_name}'

# Download annotation files and additional data
download_file(f'{url}.atr', f'{filepath}.atr')
download_file(f'{url}.qrs', f'{filepath}.qrs')
download_file(f'https://physionet.org/files/shdb-af/1.0.1/AdditionalData.csv', f'{data_path}/AdditionalData.csv')


### 4. Explanation of SHDB-AF Dataset

Key Components of SHDB-AF:

- ECG signals: These are the raw ECG waveform data recorded over time (.dat files).

- R-peak annotations: Each R-peak is labeled to indicate whether it corresponds to an AF event (1) or non-AF (0) (.atr files).

- Overall patient label: Each patient has a label indicating their overall AF status: PAF (Paroxysmal AF), Persistent AF, or Non-AF.

### 5. Preprocess ECG Data and Create Windows

Now, we preprocess the ECG data by extracting R-peaks and calculating RR intervals. We will then create fixed-length windows (e.g., 60-beat windows) and associate labels for training.

Preprocessing Steps:

- Extract RR intervals: Calculate the time difference between consecutive R-peaks.

- Create windows: Split the R-peaks data into 60-beat windows.

- Assign labels: Each window will be labeled based on the presence of AF in the peaks.

In [5]:
# Load annotations (e.g., R-peaks) and patient data
annotation = wfdb.rdann(filepath, 'atr')
additionaldata = pd.read_csv(f'{data_path}/AdditionalData.csv')

In [6]:
# Extract R-peak positions and calculate RR intervals
r_peaks = annotation.sample
rr_intervals = np.diff(r_peaks).astype(np.float32)  # Calculate RR intervals in samples
rr_time = r_peaks[1:] / annotation.fs  # Convert sample indices to time (seconds)
rr_data = rr_intervals / annotation.fs  # RR intervals in seconds

# Define window size: 60 beats per window (you can adjust this)
window_size = 60  # 60 beats per window
num_windows = len(rr_data) // window_size  # Number of 60-beat windows

# Split the RR data into windows
windows_rr = rr_data[:num_windows * window_size].reshape(num_windows, window_size)
windows_ts = rr_time[:num_windows * window_size].reshape(num_windows, window_size)


### 6. Process Preceding Windows

Here, we define a function calc_preceding_windows to calculate preceding windows based on the RR intervals. This function will be used to create additional features, which is used during training to capture temporal information in the data.

In [11]:
def calc_preceding_windows(annotations, fs, window_size):
    """
    Calculate preceding windows based on R-peak annotations.

    :param annotations: R-peak annotations.
    :param fs: Sampling frequency.
    :param window_size: Size of the window in beats.
    :returns: Masked windows to exclude certain portions of the data.
    """
    start_rr, end_rr = annotations[:-1] / fs, annotations[1:] / fs
    interbeats = np.append(np.insert((start_rr + end_rr) / 2, 0, max(0, start_rr[0] - 1)), end_rr[-1] + 1.0)
    excluded_portions_dict = np.array([[0, start_rr[0]]])

    # Split the data into windows and apply the mask
    start_win = interbeats[:-2][:(len(rr_data) // window_size) * window_size].reshape(-1, window_size)[:,
                0]  # [:, 0] to select the beginning of the window
    end_win = interbeats[2:][:(len(rr_data) // window_size) * window_size].reshape(-1, window_size)[:,
              -1]  # [:, -1] to select the end of the window
    mask_start = np.logical_or.reduce(tuple([np.logical_and(start_win > x[0], start_win <= x[1]) for x in excluded_portions_dict]))  # The window begins in an excluded portion.
    mask_end = np.logical_or.reduce(tuple([np.logical_and(end_win > x[0], end_win <= x[1]) for x in
                                           excluded_portions_dict]))  # The window ends in an excluded portion.
    mask_between = np.logical_or.reduce(tuple([np.logical_and(start_win <= x[0], end_win > x[1]) for x in excluded_portions_dict]))  # The window contains an excluded portion.
    final_mask = np.logical_not(
        np.logical_or.reduce((mask_start, mask_end, mask_between)))

    return np.cumsum(final_mask)  # Return cumulative sum mask

prec_windows = calc_preceding_windows(r_peaks, annotation.fs, window_size)

### 7. Generate Labels

We generate the labels for each window based on the AF events present in the peaks. The AF label is 1 if the more than half of the R-peaks in the window are labeled as AF, and 0 otherwise.

In [36]:
def pad_rhythm(rhythm, missing=None):
    """
    Helper function which receives the changes in the cardiac rhythm labels and pads the whole vector.
        Example:
            in = ['AFIB', '', '', '', '', '', 'N', '', '', '', 'SBR', '']
            out = ['AFIB', 'AFIB', 'AFIB', 'AFIB', 'AFIB', 'AFIB', 'N', 'N', 'N', 'N', 'SBR', 'SBR']
        This function in used to parse the '.bea' files summarizing the beats detected in the UVAF database.
    :param rhythm: The input vector representing the changes in the cardiac rhythm (list of strings or labels).
    :param missing: The different strings or labels (list) which characterize a missing rhythm. (If None, considering only '' as a missing rhythm)
    :returns rhythm: The padded vector of rhythms.
    """
    cond = np.ones(len(rhythm), dtype=bool)
    if missing != None:
        for char in missing:
            cond = np.logical_and(cond, rhythm != char)
    else:
        cond = rhythm != missing
    not_none = np.where(cond)[0]
    if len(not_none) == 0:
        rhythm = np.array(['(N'] * len(rhythm))
    else:
        not_none = np.append(not_none, len(rhythm))
        diffs = np.diff(not_none)
        sing_rhy = rhythm[not_none[:-1]]
        rhythm[not_none[0]:] = np.repeat(sing_rhy, diffs)
        rhythm[0:not_none[0]] = sing_rhy[0]
    return rhythm


def calc_y(rhythm, window_size):
    """
    Generates window labels based on the rhythm sequence.

    :param rhythm: The sequence of rhythm labels (AF or non-AF).
    :param window_size: The size of each window in beats.
    :returns: Binary array with labels (1 for AF, 0 for non-AF).
    """
    rhythms = pad_rhythm(np.array(rhythm), missing=['', 'None'])
    rlab = rhythms[:(len(rhythms) // window_size) * window_size].reshape(-1, window_size)
    counts = np.sum((rlab == '(AFIB'), axis=1)
    return counts >= window_size // 2  # If half or more are AF, label as AF

y = calc_y(annotation.aux_note, window_size)

### 8. Create Dataset for Model Training

Now we prepare the dataset by concatenating all features into a single matrix X and the labels into the y vector. We'll also include additional patient-level features such as the patient ID and global label.

In [15]:
# Generate global labels based on the patient diagnosis
matching_row = additionaldata[additionaldata['Data_ID'].astype(str).str.zfill(3).eq(record_name)]
global_label_dict = {'non-AF': 0, 'PAF': 1, 'PerAF': 3}
global_labels = np.repeat(global_label_dict[matching_row.AF_Type[0]], num_windows)

# Add patient IDs
patient_id = np.array(np.repeat(record_name, num_windows), dtype='str')

# Combine windows, preceding windows, labels, and patient IDs
X = np.concatenate((windows_rr.astype(np.float32), prec_windows.reshape(-1, 1), global_labels.astype(np.float32).reshape(-1, 1), patient_id.reshape(-1, 1)), axis=1)

# Add timestamps for each window (start and end times)
win_start = windows_ts[:, 0]
win_end = windows_ts[:, -1]
timestamp = np.concatenate((win_start.reshape(-1, 1), win_end.reshape(-1, 1)), axis=1)


In [16]:
final_data = (X, y, timestamp)
training_file_path = os.path.join(output_path, 'training_data.pickle')
with open(training_file_path, 'wb') as f:
    pickle.dump(final_data, f)

print("Data for ArNet2 model prepared and saved as training_data.pickle")


Data for ArNet2 model prepared and saved as training_data.pickle


### 9. Train ArNet2 Model

Now that we have preprocessed the data, we will train the ArNet2 model.

In [40]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
config_file_path = os.path.join(project_root, 'config', 'config.yml')
import os
import yaml

# Assuming config.yml is in the ./config/ directory of your project
config_file_path = os.path.join(project_root, 'config', 'config.yml')

# Load the config file (YAML format assumed)
with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

# Convert relative paths to absolute paths based on project_root
config['path']['arnet2'] = os.path.abspath(os.path.join(project_root, config['path']['arnet2']))
config['path']['resnet'] = os.path.abspath(os.path.join(project_root, config['path']['resnet']))

# Save the updated config back to the file
updated_config_file_path = os.path.join(project_root, 'config', 'config_w_abs_path.yml')

with open(updated_config_file_path, 'w') as file:
    yaml.dump(config, file, default_flow_style=False)

subprocess.run(['python', '../run_ArNet2.py', '--mode', 'train', '--input_file', training_file_path, '--output_name', 'ArNet2', '--save_model_path', '../model/', '--config', updated_config_file_path])

Traceback (most recent call last):
  File "../run_ArNet2.py", line 9, in <module>
    import model_utils as model_utils
  File "/home/shanybiton/repos/Shany_Repo/ArNet2/src/models/model_utils.py", line 5, in <module>
    import h5py
ModuleNotFoundError: No module named 'h5py'


CompletedProcess(args=['python', '../run_ArNet2.py', '--mode', 'train', '--input_file', '.././data/training_data.pickle', '--output_name', 'ArNet2', '--save_model_path', '../model/', '--config', '/home/shanybiton/repos/Shany_Repo/plug-and-play/config/config_w_abs_path.yml'], returncode=1)

### 10. Summary

This notebook demonstrated the following steps:

- Preprocessing the SHDB-AF dataset (R-peak annotations, RR intervals, windows).

- Training and saving the ArNet2 model for AF classification.

You can now use the trained model to predict AF in ECG signals from new patients.